In [4]:
# =============================================

"""
Convierte archivos .xls (Excel 97) a .xlsx preservando formato original.
Requiere: pip install openpyxl xlrd
"""

import os
import glob
from pathlib import Path
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import xlrd

# =============================================
#  CONFIGURA AQUÍ TUS CARPETAS
# =============================================
ANOMES = 202512

CARPETAS = [
    Path(rf"C:\Users\KOT14\Documents\Integral\Marine\Insumos\{ANOMES}"),
    Path(rf"C:\Users\KOT14\Documents\Integral\Insumos\Contabilidad\{ANOMES}"),
]
# =============================================

XL_COLORS = {
    0:  "000000", 1:  "FFFFFF", 2:  "FF0000", 3:  "00FF00",
    4:  "0000FF", 5:  "FFFF00", 6:  "FF00FF", 7:  "00FFFF",
    8:  "800000", 9:  "008000", 10: "000080", 11: "808000",
    12: "800080", 13: "008080", 14: "C0C0C0", 15: "808080",
    16: "9999FF", 17: "993366", 18: "FFFFCC", 19: "CCFFFF",
    20: "660066", 21: "FF8080", 22: "0066CC", 23: "CCCCFF",
    24: "000080", 25: "FF00FF", 26: "FFFF00", 27: "00FFFF",
    28: "800080", 29: "800000", 30: "008080", 31: "0000FF",
    32: "00CCFF", 33: "CCFFFF", 34: "CCFFCC", 35: "FFFF99",
    36: "99CCFF", 37: "FF99CC", 38: "CC99FF", 39: "FFCC99",
    40: "3366FF", 41: "33CCCC", 42: "99CC00", 43: "FFCC00",
    44: "FF9900", 45: "FF6600", 46: "666699", 47: "969696",
    48: "003366", 49: "339966", 50: "003300", 51: "333300",
    52: "993300", 53: "993366", 54: "333399", 55: "333333",
    64: "000000", 65: "FFFFFF",
}

H_ALIGN = {1: "left", 2: "left", 3: "right", 4: "right",
           5: "center", 6: "center", 7: "justify"}
V_ALIGN = {0: "top", 1: "center", 2: "bottom",
           3: "justify", 4: "distributed"}
BORDER_STYLE = {
    1: "thin", 2: "medium", 3: "dashed", 4: "dotted",
    5: "thick", 6: "double", 7: "hair", 8: "mediumDashed",
    9: "dashDot", 10: "mediumDashDot", 11: "dashDotDot",
    12: "mediumDashDotDot", 13: "slantDashDot",
}

def get_color(idx):
    if idx is None or idx == 0x7FFF:
        return None
    return XL_COLORS.get(idx)

def make_side(line_type, color_idx):
    style = BORDER_STYLE.get(line_type)
    if not style:
        return Side()
    color = get_color(color_idx)
    return Side(border_style=style, color=color or "000000")

def convertir_xls_a_xlsx(ruta_xls):
    ruta_xlsx = str(Path(ruta_xls).with_suffix(".xlsx"))
    try:
        wb_in  = xlrd.open_workbook(ruta_xls, formatting_info=True)
        wb_out = Workbook()
        wb_out.remove(wb_out.active)

        for sh_idx in range(wb_in.nsheets):
            ws_in  = wb_in.sheet_by_index(sh_idx)
            ws_out = wb_out.create_sheet(title=ws_in.name)

            # Anchos de columna
            for col_idx in range(ws_in.ncols):
                col_info = ws_in.colinfo_map.get(col_idx)
                if col_info:
                    ws_out.column_dimensions[get_column_letter(col_idx + 1)].width = max(col_info.width / 256, 1)

            # Altos de fila
            for row_idx in range(ws_in.nrows):
                row_info = ws_in.rowinfo_map.get(row_idx)
                if row_info:
                    ws_out.row_dimensions[row_idx + 1].height = row_info.height / 20

            # Celdas
            for row_idx in range(ws_in.nrows):
                for col_idx in range(ws_in.ncols):
                    cell_in  = ws_in.cell(row_idx, col_idx)
                    cell_out = ws_out.cell(row=row_idx + 1, column=col_idx + 1)

                    # Valor
                    ctype = cell_in.ctype
                    val   = cell_in.value
                    if ctype == xlrd.XL_CELL_DATE:
                        try:
                            import datetime
                            val = xlrd.xldate_as_datetime(val, wb_in.datemode)
                        except Exception:
                            pass
                    elif ctype == xlrd.XL_CELL_BOOLEAN:
                        val = bool(val)
                    elif ctype == xlrd.XL_CELL_ERROR:
                        val = None
                    cell_out.value = val

                    # Formato numérico
                    xf_idx = ws_in.cell_xf_index(row_idx, col_idx)
                    xf     = wb_in.xf_list[xf_idx]
                    fmt    = wb_in.format_map.get(xf.format_key)
                    if fmt and fmt.format_str not in ("General", "@", ""):
                        cell_out.number_format = fmt.format_str

                    # Fuente
                    font_rec = wb_in.font_list[xf.font_index]
                    color    = get_color(font_rec.colour_index)
                    cell_out.font = Font(
                        name      = font_rec.name,
                        size      = font_rec.height / 20,
                        bold      = bool(font_rec.bold),
                        italic    = bool(font_rec.italic),
                        underline = "single" if font_rec.underline_type else None,
                        strike    = bool(font_rec.struck_out),
                        color     = color,
                    )

                    # Relleno
                    bg_idx = xf.background.pattern_colour_index
                    bg_col = get_color(bg_idx)
                    pat    = xf.background.pattern_type
                    if bg_col and pat:
                        cell_out.fill = PatternFill(
                            fill_type   = "solid",
                            start_color = bg_col,
                            end_color   = bg_col,
                        )

                    # Alineación
                    al = xf.alignment
                    cell_out.alignment = Alignment(
                        horizontal = H_ALIGN.get(al.hor_align, "general"),
                        vertical   = V_ALIGN.get(al.vert_align, "bottom"),
                        wrap_text  = bool(al.text_wrapped),
                    )

                    # Bordes
                    brd = xf.border
                    cell_out.border = Border(
                        left   = make_side(brd.left_line_type,   brd.left_colour_index),
                        right  = make_side(brd.right_line_type,  brd.right_colour_index),
                        top    = make_side(brd.top_line_type,    brd.top_colour_index),
                        bottom = make_side(brd.bottom_line_type, brd.bottom_colour_index),
                    )

            # Celdas combinadas
            for rlo, rhi, clo, chi in ws_in.merged_cells:
                ws_out.merge_cells(
                    start_row=rlo + 1, end_row=rhi,
                    start_column=clo + 1, end_column=chi,
                )

        wb_out.save(ruta_xlsx)
        print(f"  ✅ Convertido: {Path(ruta_xls).name}")
        return True

    except Exception as e:
        print(f"  ❌ Error en {Path(ruta_xls).name}: {e}")
        return False


def main():
    total    = 0
    exitosos = 0

    for carpeta in CARPETAS:
        archivos = glob.glob(str(carpeta / "*.xls"))
        if not archivos:
            print(f"\n📁 {carpeta}\n   (no se encontraron archivos .xls)")
            continue

        print(f"\n📁 {carpeta}  —  {len(archivos)} archivo(s) encontrado(s)")
        for archivo in archivos:
            total += 1
            if convertir_xls_a_xlsx(archivo):
                exitosos += 1

    print(f"\n{'='*50}")
    print(f"Resultado: {exitosos}/{total} archivos convertidos correctamente.")
    print("Los .xlsx se guardaron en la misma carpeta que los originales.")


if __name__ == "__main__":
    main()


📁 C:\Users\KOT14\Documents\Integral\Marine\Insumos\202512  —  5 archivo(s) encontrado(s)
  ❌ Error en BDX Pemex NCGL-070-1002258 REAS DICIEMBRE 2025.xls: 'XFBackground' object has no attribute 'pattern_type'
  ❌ Error en Bordereaux_Inbursa_Casco_Pandi_DICIEMBRE_2025.xls: 'XFBackground' object has no attribute 'pattern_type'
  ❌ Error en Bordereaux_Inbursa_Casco_Pandi_Transportes_13-15_DICIEMBRE_2025.xls: 'XFBackground' object has no attribute 'pattern_type'
  ❌ Error en Bordereaux_Inbursa_Casco_Pandi_Transporte_09-11_DICIEMBRE_2025.xls: 'XFBackground' object has no attribute 'pattern_type'
  ❌ Error en Bordereaux_Inbursa_Casco_Pandi_Transporte_11-13_DICIEMBRE_2025.xls: 'XFBackground' object has no attribute 'pattern_type'

📁 C:\Users\KOT14\Documents\Integral\Insumos\Contabilidad\202512
   (no se encontraron archivos .xls)

Resultado: 0/5 archivos convertidos correctamente.
Los .xlsx se guardaron en la misma carpeta que los originales.
